
*  DSC 530
*  Week 9 & 10 Assignment
*  Saebastion Cole

# Hands-on data analysis with Pandas

In [103]:
# Before I do anything I am starting with the imports.
# I am opting to stick with pandas and sklearn as these are what was used in the book.

import numpy as np
import pandas as pd

# Import the modeling libraries
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import fowlkes_mallows_score

# I will also load the data from the csv here.
red_wine = pd.read_csv("data/winequality-red.csv")

# This file from github for some reason wasn't actually in csv format. it was seperated by semicolons. I will use the sep argument to fix this.
white_wine = pd.read_csv("data/winequality-white.csv", sep=";")

# Chapter 9 Exercises

## Exercise 1: Clustering Model

1. Combine the red and white wine datasets (data/winequality-red.csv and data/winequality-white.csv, respectively) and add a column for the kind of wine (red or white).

In [104]:
# Actually doing this backwards to how it was asked because we lose what the source was once we combine them.


# Add the new column with the type.
red_wine["kind_of_wine"] = "red"
white_wine["kind_of_wine"] = "white"

# Concat both of the dataframes into the one.
wine = pd.concat([red_wine, white_wine], ignore_index=True)
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,kind_of_wine
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


2. Build and fit a pipeline that scales the data and then uses k-means clustering to make two clusters. Be sure not to use the quality column.

In [105]:
# Combine the two dataframes again

wine = pd.concat([red_wine, white_wine], ignore_index=True)

kmeans_wine_data = wine.copy()

kmeans_wine_data.drop(columns=["kind_of_wine"], inplace=True)
kmeans_wine_data.drop(columns=["quality"], inplace=True)

kmeans_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=2, random_state=42))])

kmeans_pipeline.fit(kmeans_wine_data)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('kmeans', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['fixed acidity','volatile acidity','citric acid',...,'pH','sulphates', 'alcohol']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [106]:
kmeans_wine_data["cluster"] = kmeans_pipeline.predict(kmeans_wine_data)
kmeans_wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,cluster
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0


3. Use the Fowlkes-Mallows Index (the fowlkes_mallows_score() function is in sklearn.metrics) to evaluate how well k-means is able to make the distinction between red and white wine.

In [107]:
fowlkes_mallows_score(wine["kind_of_wine"], kmeans_wine_data["cluster"])

0.9776918113120894

4. Find the center of each cluster.

In [108]:
kmeans = kmeans_pipeline.named_steps["kmeans"]
kmeans.cluster_centers_

# We need to get the columns from the original data to see where the center exists.
feature_columns = kmeans_wine_data.columns.drop("cluster")

scaler = kmeans_pipeline.named_steps["scaler"]

# Take the centers of each column for each cluster and put it into a data frame
centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=feature_columns
)
centers


# Add column name for the colors of wine we are using for the clusters
centers.index = ["red", "white"]

# Show the data
centers.T

,red,white
fixed acidity,8.288200,6.851855
volatile acidity,0.531819,0.274572
citric acid,0.269544,0.335263
residual sugar,2.633151,6.395178
chlorides,0.088383,0.045075
free sulfur dioxide,15.762470,35.526375
total sulfur dioxide,48.724453,138.448279
density,0.996738,0.994005
pH,3.309605,3.187639
sulphates,0.656697,0.488778


## Exercise 2.

Create a histogram and KDE of the change from open to close in the price of Facebook stock.